In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parent if Path212.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

import os
import getpass

if not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass("Enter Groq API Key: ")
if not os.environ.get('GOOGLE_API_KEY'):
    os.environ['GOOGLE_API_KEY'] = getpass.getpass("Enter Google API Key: ")
if not os.environ('LANGCHAIN_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = getpass.getpass("Enter LangSmith API Key: ")
    
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT'] = 'rag-lab-query-translation'

In [2]:
# Loading the Vectorstore
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma   
from langchain_google_genai import ChatGoogleGenerativeAI

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(
    embedding_function=embeddings,
    persist_directory=str(project_root / "data" / "chroma_naive_gemini")
)
print("Chunks in store:", vectorstore._collection.count())

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

Chunks in store: 979


In [3]:
# Reciprocal Rank Function

from typing import List
from langchain_core.documents import Document


def reciprocal_rank_fusion(
    doc_lists: List[List[Document]], k: int = 60
) -> List[Document]:
    """Merge multiple ranked document lists into one, scored by RRF.
    Documents appearing consistently near the top across lists win,
    even if no single list ranked them #1.
    """
    fused_scores: dict[str, float] = {}
    doc_lookup: dict[str, Document] = {}

    for docs in doc_lists:
        for rank, doc in enumerate(docs):
            key = doc.page_content  # dedup key, same approach as multi-query
            doc_lookup[key] = doc
            fused_scores[key] = fused_scores.get(key, 0.0) + 1.0 / (k + rank + 1)

    reranked_keys = sorted(fused_scores, key=lambda x: fused_scores[x], reverse=True)
    return [doc_lookup[key] for key in reranked_keys]

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStore

# reuse from multi_query.py — import rather than redefine, once extracted
from rag_lab.strategies.multi_query import LineListOutputParser, MULTI_QUERY_PROMPT


class RagFusionStrategy:
    """Same query-rephrasing step as MultiQueryStrategy, but merges
    retrieved documents via Reciprocal Rank Fusion instead of a
    flat unique union — rewarding documents ranked highly across
    multiple query variants."""

    def __init__(self, vectorstore: VectorStore, llm: BaseChatModel, k: int = 4, rrf_k: int = 60):
        self.retriever = vectorstore.as_retriever(search_kwargs={"k": k})
        self.llm = llm
        self.rrf_k = rrf_k
        self.query_gen_chain = MULTI_QUERY_PROMPT | llm | LineListOutputParser()
        self.answer_prompt = ChatPromptTemplate.from_template(
            "Answer the question based only on the following context:\n"
            "{context}\n\nQuestion: {question}"
        )

    def generate_queries(self, query: str) -> list[str]:
        return [query] + self.query_gen_chain.invoke({"question": query})

    def retrieve(self, query: str) -> list[Document]:
        all_queries = self.generate_queries(query)
        doc_lists = [self.retriever.invoke(q) for q in all_queries]
        return reciprocal_rank_fusion(doc_lists, k=self.rrf_k)

    def run(self, query: str) -> str:
        docs = self.retrieve(query)
        context = "\n\n".join(d.page_content for d in docs)
        chain = self.answer_prompt | self.llm | StrOutputParser()
        return chain.invoke({"context": context, "question": query})

In [5]:
# Testing

from rag_lab.strategies.multi_query import MultiQueryStrategy
from rag_lab.strategies.rag_fusion import RagFusionStrategy

test_query = "Why do PINNs struggle with irregular geometry?"

mq_strategy = MultiQueryStrategy(vectorstore, llm=llm)
rf_strategy = RagFusionStrategy(vectorstore, llm=llm)

mq_docs = mq_strategy.retrieve(test_query)
rf_docs = rf_strategy.retrieve(test_query)

print(f"Multi-query retrieved {len(mq_docs)} unique docs")
print(f"RAG-Fusion retrieved {len(rf_docs)} unique docs, ranked by RRF score\n")

print("Top 3 RAG-Fusion results:")
for i, doc in enumerate(rf_docs[:3]):
    print(f"{i+1}. {doc.page_content[:150]}...\n")

/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.
/home/mohnish/my-jupyter-env/lib/python3.14/site-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Multi-query retrieved 4 unique docs
RAG-Fusion retrieved 4 unique docs, ranked by RRF score

Top 3 RAG-Fusion results:
1. integrates the underlying physical law described by PDEs with fully connected (FC)
networks. This work lays a solid foundation for solving both forwar...

2. introduced a gradient -free physics -informed learning method based on random projections, 
successfully simulating fourth-order phase-field fracture ...

3. The results demonstrate that, under identical configurations, SK -PINN is several times faster 
than AD-PINN and mAD-PINN, while achieving the same le...

